# Healthcare Diabetes — ML Classification Project
**Author:** Jahid Shahid Patel  
**Dataset:** [Healthcare Diabetes — Kaggle](https://www.kaggle.com/datasets/nanditapore/healthcare-diabetes)  
**Goal:** Predict diabetes risk (binary classification) from clinical parameters using Random Forest & Logistic Regression.

---
### Pipeline Overview
1. Data Loading & Exploration
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Model Training (Random Forest + Logistic Regression)
5. Model Evaluation
6. Save Model
7. Streamlit Web UI (interactive prediction app — run via `streamlit run`)

## 1. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for notebook compatibility
import seaborn as sns
import warnings
import joblib
import os

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
print('All libraries imported successfully.')

All libraries imported successfully.


## 2. Data Loading & Exploration

In [2]:
df = pd.read_csv('Healthcare-Diabetes.csv')

# Drop the Id column — not a clinical feature
if 'Id' in df.columns:
    df.drop(columns=['Id'], inplace=True)

print('Dataset shape:', df.shape)
print('\nColumn names:', df.columns.tolist())
df.head()

Dataset shape: (2768, 9)

Column names: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
print('=== Dataset Info ===')
df.info()
print('\n=== Statistical Summary ===')
df.describe().round(2)

=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2768 entries, 0 to 2767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               2768 non-null   int64  
 1   Glucose                   2768 non-null   int64  
 2   BloodPressure             2768 non-null   int64  
 3   SkinThickness             2768 non-null   int64  
 4   Insulin                   2768 non-null   int64  
 5   BMI                       2768 non-null   float64
 6   DiabetesPedigreeFunction  2768 non-null   float64
 7   Age                       2768 non-null   int64  
 8   Outcome                   2768 non-null   int64  
dtypes: float64(2), int64(7)
memory usage: 194.8 KB

=== Statistical Summary ===


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,2768.00,2768.00,2768.00,2768.00,2768.00,2768.00,2768.00,2768.00,2768.00
mean,3.74,121.10,69.13,20.82,80.13,32.14,0.47,33.13,0.34
std,3.32,32.04,19.23,16.06,112.30,8.08,0.33,11.78,0.48
min,0.00,0.00,0.00,0.00,0.00,0.00,0.08,21.00,0.00
25%,1.00,99.00,62.00,0.00,0.00,27.30,0.24,24.00,0.00
50%,3.00,117.00,72.00,23.00,37.00,32.20,0.38,29.00,0.00
75%,6.00,141.00,80.00,32.00,130.00,36.62,0.62,40.00,1.00
max,17.00,199.00,122.00,110.00,846.00,80.60,2.42,81.00,1.00


In [4]:
print('Missing values per column:')
print(df.isnull().sum())

print('\nClass distribution (Outcome):')
print(df['Outcome'].value_counts())
print(f'  Diabetic (1): {df["Outcome"].sum()} | Non-Diabetic (0): {(df["Outcome"]==0).sum()}')

Missing values per column:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Class distribution (Outcome):
Outcome
0    1816
1     952
Name: count, dtype: int64
  Diabetic (1): 952 | Non-Diabetic (0): 1816


## 3. Exploratory Data Analysis (EDA)

In [5]:
# --- 3.1 Class Distribution ---
fig, ax = plt.subplots(figsize=(5, 4))
counts = df['Outcome'].value_counts()
ax.bar(['Non-Diabetic (0)', 'Diabetic (1)'], counts.values,
       color=['#3b82d4', '#ef4444'], edgecolor='white', linewidth=0.8)
ax.set_title('Class Distribution — Outcome', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120)
plt.show()
print('Saved: class_distribution.png')

Saved: class_distribution.png


In [6]:
# --- 3.2 Feature Distributions (Histograms) ---
features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
            'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(features):
    axes[i].hist(df[col], bins=25, color='#3b82d4', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
# Hide empty subplot
axes[-1].set_visible(False)
fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: feature_distributions.png')

Saved: feature_distributions.png


In [7]:
# --- 3.3 Box Plots by Outcome ---
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(features):
    df.boxplot(column=col, by='Outcome', ax=axes[i],
               boxprops=dict(color='#3b82d4'),
               medianprops=dict(color='#ef4444', linewidth=2))
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('Outcome (0=No, 1=Yes)')
plt.suptitle('Feature Box Plots by Diabetes Outcome', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplots_by_outcome.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: boxplots_by_outcome.png')

Saved: boxplots_by_outcome.png


In [8]:
# --- 3.4 Correlation Heatmap ---
fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: correlation_heatmap.png')

Saved: correlation_heatmap.png


## 4. Data Preprocessing

In [9]:
# --- 4.1 Replace biologically invalid zeros with NaN (impute later) ---
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df_clean = df.copy()
for col in zero_cols:
    df_clean[col] = df_clean[col].replace(0, np.nan)

print('Zero → NaN replacement done.')
print('\nMissing values after replacement:')
print(df_clean.isnull().sum())

Zero → NaN replacement done.

Missing values after replacement:
Pregnancies                    0
Glucose                       18
BloodPressure                125
SkinThickness                800
Insulin                     1330
BMI                           39
DiabetesPedigreeFunction       0
Age                            0
Outcome                        0
dtype: int64


In [10]:
# --- 4.2 Median Imputation ---
X = df_clean.drop(columns=['Outcome'])
y = df_clean['Outcome']

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)
X_imputed = pd.DataFrame(X_imputed, columns=X.columns)

print('Imputation complete. Missing values remaining:', X_imputed.isnull().sum().sum())

Imputation complete. Missing values remaining: 0


In [11]:
# --- 4.3 Train / Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

Train size: 2214 | Test size: 554


In [12]:
# --- 4.4 Feature Scaling ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('Scaling complete.')

Scaling complete.


## 5. Model Training

In [13]:
# --- 5.1 Random Forest ---
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    class_weight='balanced'
)
rf_model.fit(X_train_scaled, y_train)
print('Random Forest trained.')

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rf_cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print(f'RF Cross-Val Accuracy: {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}')

Random Forest trained.


RF Cross-Val Accuracy: 0.8902 ± 0.0119


In [14]:
# --- 5.2 Logistic Regression ---
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    class_weight='balanced'
)
lr_model.fit(X_train_scaled, y_train)
print('Logistic Regression trained.')

lr_cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print(f'LR Cross-Val Accuracy: {lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}')

Logistic Regression trained.
LR Cross-Val Accuracy: 0.7574 ± 0.0130


## 6. Model Evaluation

In [15]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_proba)
    
    print(f'\n=== {model_name} — Test Set Results ===')
    print(f'  Accuracy : {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall   : {rec:.4f}')
    print(f'  F1 Score : {f1:.4f}')
    print(f'  ROC-AUC  : {auc:.4f}')
    print(f'\n{classification_report(y_test, y_pred, target_names=["Non-Diabetic","Diabetic"])}')
    return y_pred, y_proba, {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'auc': auc}

rf_pred, rf_proba, rf_metrics = evaluate_model(rf_model, X_test_scaled, y_test, 'Random Forest')
lr_pred, lr_proba, lr_metrics = evaluate_model(lr_model, X_test_scaled, y_test, 'Logistic Regression')


=== Random Forest — Test Set Results ===
  Accuracy : 0.9152
  Precision: 0.8303
  Recall   : 0.9476
  F1 Score : 0.8851
  ROC-AUC  : 0.9782



              precision    recall  f1-score   support

Non-Diabetic       0.97      0.90      0.93       363
    Diabetic       0.83      0.95      0.89       191

    accuracy                           0.92       554
   macro avg       0.90      0.92      0.91       554
weighted avg       0.92      0.92      0.92       554


=== Logistic Regression — Test Set Results ===
  Accuracy : 0.7599
  Precision: 0.6394
  Recall   : 0.6963
  F1 Score : 0.6667
  ROC-AUC  : 0.8347

              precision    recall  f1-score   support

Non-Diabetic       0.83      0.79      0.81       363
    Diabetic       0.64      0.70      0.67       191

    accuracy                           0.76       554
   macro avg       0.74      0.74      0.74       554
weighted avg       0.77      0.76      0.76       554



In [16]:
# --- 6.1 Confusion Matrices ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, pred, name in zip(axes, [rf_pred, lr_pred], ['Random Forest', 'Logistic Regression']):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non-Diabetic', 'Diabetic'],
                yticklabels=['Non-Diabetic', 'Diabetic'],
                linewidths=0.5)
    ax.set_title(f'{name} — Confusion Matrix', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices.png')

Saved: confusion_matrices.png


In [17]:
# --- 6.2 ROC Curves ---
fig, ax = plt.subplots(figsize=(7, 6))
for proba, name, color in [
    (rf_proba, 'Random Forest', '#3b82d4'),
    (lr_proba, 'Logistic Regression', '#ef4444')
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_val = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', color=color, linewidth=2)

ax.plot([0,1],[0,1], 'k--', linewidth=1, label='Random Baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: roc_curves.png')

Saved: roc_curves.png


In [18]:
# --- 6.3 Feature Importance (Random Forest) ---
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.plot(kind='barh', ax=ax, color='#3b82d4', edgecolor='white')
ax.set_title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: feature_importance.png')
print('\nTop 3 most important features:')
print(feat_imp.sort_values(ascending=False).head(3))

Saved: feature_importance.png

Top 3 most important features:
Glucose    0.304005
BMI        0.179217
Age        0.148116
dtype: float64


In [19]:
# --- 6.4 Model Comparison Bar Chart ---
metrics_df = pd.DataFrame([rf_metrics, lr_metrics],
                          index=['Random Forest', 'Logistic Regression'])
fig, ax = plt.subplots(figsize=(9, 5))
metrics_df[['accuracy','precision','recall','f1','auc']].plot(
    kind='bar', ax=ax, edgecolor='white', width=0.7
)
ax.set_title('Model Comparison — Key Metrics', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim([0, 1.05])
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: model_comparison.png')

Saved: model_comparison.png


## 7. Save Model Artifacts

In [20]:
joblib.dump(rf_model,  'rf_model.pkl')
joblib.dump(lr_model,  'lr_model.pkl')
joblib.dump(scaler,    'scaler.pkl')
joblib.dump(imputer,   'imputer.pkl')
print('Model artifacts saved:')
print('  rf_model.pkl, lr_model.pkl, scaler.pkl, imputer.pkl')

Model artifacts saved:
  rf_model.pkl, lr_model.pkl, scaler.pkl, imputer.pkl


---
## 8. Streamlit Web UI

The cell below writes the Streamlit app to `app.py`.  
After running this notebook, launch the Web UI with:

```bash
streamlit run app.py
```

In [21]:
streamlit_code = '''
# ============================================================
# Healthcare Diabetes — Streamlit Web UI
# Author: Jahid Shahid Patel
# Run: streamlit run app.py
# ============================================================
import streamlit as st
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import os

# --- Page Config ---
st.set_page_config(
    page_title="Diabetes Risk Predictor",
    page_icon="🩺",
    layout="centered",
    initial_sidebar_state="expanded"
)

# --- Load Artifacts ---
@st.cache_resource
def load_artifacts():
    rf  = joblib.load("rf_model.pkl")
    lr  = joblib.load("lr_model.pkl")
    sc  = joblib.load("scaler.pkl")
    imp = joblib.load("imputer.pkl")
    return rf, lr, sc, imp

rf_model, lr_model, scaler, imputer = load_artifacts()

FEATURES = ["Pregnancies","Glucose","BloodPressure","SkinThickness",
            "Insulin","BMI","DiabetesPedigreeFunction","Age"]

# --- Sidebar ---
st.sidebar.image("https://img.icons8.com/color/96/000000/hospital.png", width=80)
st.sidebar.title("About")
st.sidebar.markdown("""
**Healthcare Diabetes Predictor**  
Uses clinical input features to classify diabetes risk.

- **Model:** Random Forest / Logistic Regression
- **Dataset:** [Kaggle Healthcare Diabetes](https://www.kaggle.com/datasets/nanditapore/healthcare-diabetes)
- **Author:** Jahid Shahid Patel
""")

# --- Main Title ---
st.title("🩺 Diabetes Risk Prediction")
st.markdown("Enter the patient\'s clinical parameters below to predict their diabetes risk.")
st.divider()

# --- Model Selection ---
model_choice = st.radio("Choose Model:", ["Random Forest", "Logistic Regression"], horizontal=True)
st.divider()

# --- Input Form ---
st.subheader("Patient Clinical Parameters")
col1, col2 = st.columns(2)

with col1:
    pregnancies  = st.number_input("Pregnancies",             min_value=0,   max_value=20,  value=1,    step=1)
    glucose      = st.number_input("Glucose (mg/dL)",         min_value=0,   max_value=300, value=120,  step=1)
    blood_press  = st.number_input("Blood Pressure (mm Hg)",  min_value=0,   max_value=150, value=70,   step=1)
    skin_thick   = st.number_input("Skin Thickness (mm)",     min_value=0,   max_value=100, value=20,   step=1)

with col2:
    insulin      = st.number_input("Insulin (mu U/mL)",       min_value=0,   max_value=900, value=80,   step=1)
    bmi          = st.number_input("BMI (kg/m²)",             min_value=0.0, max_value=70.0,value=32.0, step=0.1)
    dpf          = st.number_input("Diabetes Pedigree Func.", min_value=0.0, max_value=3.0, value=0.5,  step=0.01)
    age          = st.number_input("Age (years)",             min_value=1,   max_value=120, value=33,   step=1)

st.divider()

# --- Predict ---
if st.button("🔍 Predict Diabetes Risk", use_container_width=True, type="primary"):
    input_data = np.array([[pregnancies, glucose, blood_press, skin_thick,
                            insulin, bmi, dpf, age]], dtype=float)
    
    # Replace zeros in clinical columns with NaN for imputation
    zero_cols_idx = [1, 2, 3, 4, 5]  # Glucose, BP, Skin, Insulin, BMI
    for idx in zero_cols_idx:
        if input_data[0, idx] == 0:
            input_data[0, idx] = np.nan
    
    input_imputed = imputer.transform(input_data)
    input_scaled  = scaler.transform(input_imputed)
    
    model = rf_model if model_choice == "Random Forest" else lr_model
    prediction = model.predict(input_scaled)[0]
    probability = model.predict_proba(input_scaled)[0]
    
    st.subheader("Prediction Result")
    if prediction == 1:
        st.error(f"⚠️  **HIGH RISK — Diabetic**  |  Confidence: {probability[1]*100:.1f}%")
    else:
        st.success(f"✅  **LOW RISK — Non-Diabetic**  |  Confidence: {probability[0]*100:.1f}%")
    
    # Probability gauge
    st.markdown("**Probability Breakdown**")
    col_a, col_b = st.columns(2)
    col_a.metric("Non-Diabetic Probability", f"{probability[0]*100:.1f}%")
    col_b.metric("Diabetic Probability",     f"{probability[1]*100:.1f}%")
    
    # Bar chart of probabilities
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.barh(["Non-Diabetic", "Diabetic"],
            [probability[0], probability[1]],
            color=["#3b82d4", "#ef4444"], edgecolor="white")
    ax.set_xlim(0, 1)
    ax.set_xlabel("Probability")
    ax.set_title("Prediction Confidence", fontweight="bold")
    for i, v in enumerate([probability[0], probability[1]]):
        ax.text(v + 0.01, i, f"{v*100:.1f}%", va="center", fontweight="bold")
    plt.tight_layout()
    st.pyplot(fig)
    plt.close()

# --- EDA Plots (if available) ---
st.divider()
st.subheader("📊 Dataset Visualisations")
plot_files = [
    ("class_distribution.png",   "Class Distribution"),
    ("feature_distributions.png","Feature Distributions"),
    ("correlation_heatmap.png",  "Correlation Heatmap"),
    ("confusion_matrices.png",   "Confusion Matrices"),
    ("roc_curves.png",           "ROC Curves"),
    ("feature_importance.png",   "Feature Importance"),
    ("model_comparison.png",     "Model Comparison"),
]
for fname, caption in plot_files:
    if os.path.exists(fname):
        st.image(fname, caption=caption, use_column_width=True)

st.divider()
st.caption("Healthcare Diabetes ML Project — Jahid Shahid Patel")
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_code.strip())

print('app.py written successfully.')
print('To launch the Web UI, run:  streamlit run app.py')

app.py written successfully.
To launch the Web UI, run:  streamlit run app.py


---
## ✅ Pipeline Complete

| Artifact | File |
|----------|------|
| Trained Random Forest | `rf_model.pkl` |
| Trained Logistic Regression | `lr_model.pkl` |
| Scaler | `scaler.pkl` |
| Imputer | `imputer.pkl` |
| EDA Plots | `*.png` |
| Streamlit App | `app.py` |

**Run the Web UI:**
```bash
streamlit run app.py
```